In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:

from pyspark.sql import functions as F


# ---------------------------------------------------------------------------
# Source and target tables
# ---------------------------------------------------------------------------

SILVER_TABLE = (
    f"{SILVER_SCHEMA}.football_data_org_competitions"
)

GOLD_TABLE = f"{GOLD_SCHEMA}.dim_competition"


# ---------------------------------------------------------------------------
# Business keys and required columns
# ---------------------------------------------------------------------------

BUSINESS_KEY = [
    "competition_key",
]

REQUIRED_SILVER_COLUMNS = [
    "competition_id",
    "competition_code",
    "competition_name",
    "competition_type",
    "competition_emblem",
    "area_name",
    "league_code",
]

REQUIRED_GOLD_COLUMNS = [
    "competition_key",
    "competition_name",
    "competition_code",
    "country",
]


# ---------------------------------------------------------------------------
# Validate source table
# ---------------------------------------------------------------------------

if not spark.catalog.tableExists(SILVER_TABLE):
    raise ValueError(
        f"Required Silver table does not exist: {SILVER_TABLE}"
    )

silver_df = spark.table(SILVER_TABLE)

validate_columns_exist(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
)

silver_metrics = validate_required_not_null(
    silver_df,
    [
        "competition_id",
        "competition_code",
        "competition_name",
        "league_code",
    ],
    fail_on_null=True,
)

if silver_metrics["row_count"] == 0:
    raise ValueError(
        "Silver competitions dataset is empty. "
        "Refusing to build dim_competition."
    )

print(
    "Silver competition source validated: "
    f"{silver_metrics['row_count']} records."
)


# ---------------------------------------------------------------------------
# Prepare competition dimension
# ---------------------------------------------------------------------------

# Use canonical league_code as the Gold competition identifier.
# API competition IDs are source-specific and should not be used
# as the shared analytical key.

gold_df = (
    silver_df
    .select(
        F.trim(
            F.col("league_code")
        ).alias("competition_key"),

        F.trim(
            F.col("competition_name")
        ).alias("competition_name"),

        F.trim(
            F.col("competition_code")
        ).alias("competition_code"),

        F.trim(
            F.col("area_name")
        ).alias("country"),

        F.trim(
            F.col("competition_type")
        ).alias("competition_type"),

        F.trim(
            F.col("competition_emblem")
        ).alias("competition_emblem"),
    )
)


# ---------------------------------------------------------------------------
# Validate canonical competition mapping
# ---------------------------------------------------------------------------

# The shared project configuration is the authoritative source
# for supported competition codes and canonical identifiers.

expected_mapping = FOOTBALL_DATA_ORG_COMPETITIONS

expected_leagues = set(SUPPORTED_LEAGUES)

if set(expected_mapping.values()) != expected_leagues:
    raise ValueError(
        "Competition mapping does not match SUPPORTED_LEAGUES "
        "in the shared project configuration."
    )

expected_league_column = map_values(
    "competition_code",
    expected_mapping,
)

invalid_mapping = (
    gold_df
    .filter(
        expected_league_column.isNull()
        | (
            F.col("competition_key")
            != expected_league_column
        )
    )
)

if invalid_mapping.limit(1).count() > 0:
    display(invalid_mapping)

    raise ValueError(
        "Competition codes and canonical league identifiers "
        "are inconsistent."
    )


# ---------------------------------------------------------------------------
# Validate Gold competition dimension
# ---------------------------------------------------------------------------

gold_metrics = validate_required_not_null(
    gold_df,
    REQUIRED_GOLD_COLUMNS,
    fail_on_null=True,
)

gold_count = gold_metrics["row_count"]

# Each canonical competition should appear exactly once.

duplicate_keys = (
    gold_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    display(duplicate_keys)

    raise ValueError(
        "Duplicate competition_key values found "
        "in dim_competition."
    )

# API competition codes should also remain unique.

duplicate_codes = (
    gold_df
    .groupBy("competition_code")
    .count()
    .filter(F.col("count") > 1)
)

if duplicate_codes.limit(1).count() > 0:
    display(duplicate_codes)

    raise ValueError(
        "Duplicate competition_code values found "
        "in dim_competition."
    )

# Validate that all supported competitions are present
# and that no unexpected competition was included.

actual_leagues = {
    row["competition_key"]
    for row in gold_df
    .select("competition_key")
    .distinct()
    .collect()
}

missing_leagues = expected_leagues - actual_leagues
unexpected_leagues = actual_leagues - expected_leagues

if missing_leagues or unexpected_leagues:
    raise ValueError(
        "Gold competition coverage is inconsistent. "
        f"Missing leagues: {sorted(missing_leagues)}. "
        f"Unexpected leagues: {sorted(unexpected_leagues)}."
    )

# Competition names and countries must not be blank.

blank_attributes = (
    gold_df
    .filter(
        (F.length(F.col("competition_name")) == 0)
        | (F.length(F.col("competition_code")) == 0)
        | (F.length(F.col("country")) == 0)
    )
    .limit(1)
    .count()
)

if blank_attributes > 0:
    raise ValueError(
        "Blank required competition attributes "
        "found in dim_competition."
    )

print(
    "Gold validation successful: "
    f"{gold_count} unique competitions."
)


# ---------------------------------------------------------------------------
# Add Gold processing metadata
# ---------------------------------------------------------------------------

gold_df = gold_df.withColumn(
    "_gold_processed_at",
    F.current_timestamp(),
)


# ---------------------------------------------------------------------------
# Write Gold Delta table
# ---------------------------------------------------------------------------

# Full refresh is appropriate for this small dimension.
# Repeated executions overwrite existing competition records.
# The canonical competition keys remain stable across runs.

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}"
)

write_full_refresh(
    gold_df,
    GOLD_TABLE,
)


# ---------------------------------------------------------------------------
# Post-write validation
# ---------------------------------------------------------------------------

saved_df = spark.table(GOLD_TABLE)

saved_count = saved_df.count()

if saved_count != len(expected_leagues):
    raise ValueError(
        "Post-write validation failed: "
        f"expected {len(expected_leagues)} competitions, "
        f"found {saved_count}."
    )

saved_duplicate_keys = (
    saved_df
    .groupBy(*BUSINESS_KEY)
    .count()
    .filter(F.col("count") > 1)
    .limit(1)
    .count()
)

if saved_duplicate_keys > 0:
    raise ValueError(
        "Post-write validation failed: "
        "duplicate competition_key values."
    )

saved_leagues = {
    row["competition_key"]
    for row in saved_df
    .select("competition_key")
    .distinct()
    .collect()
}

if saved_leagues != expected_leagues:
    raise ValueError(
        "Post-write validation failed: "
        "competition coverage does not match configuration."
    )


# ---------------------------------------------------------------------------
# Display execution summary
# ---------------------------------------------------------------------------

print("=" * 55)
print("GOLD COMPETITION DIMENSION LOAD COMPLETED")
print("=" * 55)
print(f"Target table: {GOLD_TABLE}")
print(f"Total competitions: {saved_count}")
print(f"Unique competition keys: {len(saved_leagues)}")
print(f"Duplicate competition keys: {saved_duplicate_keys}")
print("=" * 55)

display(
    saved_df.orderBy("competition_name")
)
